# Baseline: приоритизация обращений

Минимальный baseline для задачи.  
Он обучает простую модель только на готовых табличных признаках из `train.csv` / `test.csv` и создает `submission.csv`.

`events.csv` можно использовать для улучшения решения, но в baseline признаки из событий не строятся.

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Пути к данным.
ROOT = Path(".")
DATA_DIR = ROOT / "data"

TARGET = "target"

# Эти колонки не используем как признаки модели.
ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET, "split"}

RANDOM_STATE = 42

## Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.  
В baseline модель использует только `train.csv` и `test.csv`.

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)
print("target mean:", train[TARGET].mean())

train: (13694, 119)
test: (4306, 118)
events: (254705, 7)
target mean: 0.20746312253541696


## Генерация признаков из истории событий

Формируем агрегированную статистику по действиям пользователей и ценам объектов в двух разрезах: для конкретного лида и юзера.

Чтобы исключить утечку данных, учитываем только события, произошедшие до момента назначения.

Признаки рассчитываются:
- за всю доступную историю;
- внутри скользящих временных окон (последние 1, 3, 7, 14 и 30 дней).

В финальный набор включаются только общие для train и test числовые и категориальные признаки за вычетом технических колонок из NON_FEATURE_COLUMNS.

In [4]:
# Добавляем признаки из events.csv, используя только события до момента назначения.

def add_event_features(df: pd.DataFrame, events: pd.DataFrame, id_col: str) -> pd.DataFrame:
    base = df[[id_col, "assignment_ts"]].copy()
    ev = events[[id_col, "event_ts", "event_type", "item_price_log", "src_slot", "ctx_seq"]].copy()
    ev["event_type"] = ev["event_type"].astype(str)

    merged = base.merge(ev, on=id_col, how="left")
    merged = merged[merged["event_ts"] < merged["assignment_ts"]].copy()
    merged["delta_hours"] = (merged["assignment_ts"] - merged["event_ts"]).dt.total_seconds() / 3600.0

    agg = (
        merged.groupby(id_col)
        .agg(
            total_events=("event_ts", "count"),
            item_views=("event_type", lambda s: (s == "item_view").sum()),
            searches=("event_type", lambda s: (s == "search").sum()),
            favorites=("event_type", lambda s: (s == "favorite").sum()),
            chat_opens=("event_type", lambda s: (s == "chat_open").sum()),
            call_clicks=("event_type", lambda s: (s == "call_click").sum()),
            mean_price=("item_price_log", "mean"),
            max_price=("item_price_log", "max"),
            min_price=("item_price_log", "min"),
            src_slot_nunique=("src_slot", "nunique"),
            ctx_seq_nunique=("ctx_seq", "nunique"),
            last_event_hours=("delta_hours", "min"),
        )
        .reset_index()
    )

    windows = [1, 3, 7, 14, 30]
    parts = [agg]
    for w in windows:
        mask = merged["delta_hours"] <= w * 24
        tmp = (
            merged.loc[mask]
            .groupby(id_col)
            .agg(
                **{f"{id_col}_events_{w}d": ("event_ts", "count")},
                **{f"{id_col}_item_view_{w}d": ("event_type", lambda s: (s == "item_view").sum())},
                **{f"{id_col}_search_{w}d": ("event_type", lambda s: (s == "search").sum())},
                **{f"{id_col}_favorite_{w}d": ("event_type", lambda s: (s == "favorite").sum())},
                **{f"{id_col}_chat_open_{w}d": ("event_type", lambda s: (s == "chat_open").sum())},
                **{f"{id_col}_call_click_{w}d": ("event_type", lambda s: (s == "call_click").sum())},
                **{f"{id_col}_mean_price_{w}d": ("item_price_log", "mean")},
            )
            .reset_index()
        )
        parts.append(tmp)

    # Дополнительный признак: доля "полезных" событий среди всех событий за последние 7 дней.
    if merged.empty:
        useful_ratio = pd.DataFrame({id_col: base[id_col], f"{id_col}_useful_ratio_7d": 0.0})
    else:
        useful_mask = merged["delta_hours"] <= 7 * 24
        useful_events = merged.loc[useful_mask].groupby(id_col).apply(
            lambda g: ((g["event_type"].isin(["favorite", "chat_open", "call_click"])).sum() / max(1, len(g)))
        )
        useful_ratio = useful_events.rename(f"{id_col}_useful_ratio_7d").reset_index()

    parts.append(useful_ratio)

    out = base[[id_col]].copy()
    for feat in parts:
        out = out.merge(feat, on=id_col, how="left")

    return out.fillna(0)


train["assignment_ts"] = pd.to_datetime(train["assignment_ts"])
test["assignment_ts"] = pd.to_datetime(test["assignment_ts"])
events["event_ts"] = pd.to_datetime(events["event_ts"])

train_event_features = add_event_features(train, events, "lead_id")
user_event_features = add_event_features(train, events, "user_id")
test_event_features = add_event_features(test, events, "lead_id")
test_user_event_features = add_event_features(test, events, "user_id")

train_event_features = train_event_features.rename(columns=lambda c: f"lead_{c}" if c != "lead_id" else c)
user_event_features = user_event_features.rename(columns=lambda c: f"user_{c}" if c != "user_id" else c)
test_event_features = test_event_features.rename(columns=lambda c: f"lead_{c}" if c != "lead_id" else c)
test_user_event_features = test_user_event_features.rename(columns=lambda c: f"user_{c}" if c != "user_id" else c)

train = train.merge(train_event_features, on="lead_id", how="left").merge(user_event_features, on="user_id", how="left")
test = test.merge(test_event_features, on="lead_id", how="left").merge(test_user_event_features, on="user_id", how="left")

feature_columns = [
    column for column in train.columns
    if column not in NON_FEATURE_COLUMNS and column in test.columns
]

numeric_columns = [
    column for column in feature_columns
    if pd.api.types.is_numeric_dtype(train[column])
]

categorical_columns = [
    column for column in feature_columns
    if column not in numeric_columns
]

print("numeric:", len(numeric_columns))
print("categorical:", len(categorical_columns))
print("total features:", len(feature_columns))

numeric: 203
categorical: 7
total features: 210


## Валидация

Так как тестовая выборка находится позже train по времени, лучше валидироваться на последних датах train.  
Это ближе к реальному сценарию, чем случайный split.

In [5]:
def make_validation_split(train_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит train по времени: ранние даты в обучение, поздние даты в валидацию."""
    if "assignment_date" in train_df.columns:
        dates = pd.to_datetime(train_df["assignment_date"]).dt.date
        ordered_dates = sorted(dates.unique())
        cutoff = ordered_dates[int(len(ordered_dates) * 0.8)]

        train_part = train_df[dates < cutoff]
        valid_part = train_df[dates >= cutoff]
        return train_part, valid_part

    # Fallback, если даты нет.
    return train_test_split(
        train_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train_df[TARGET],
    )


train_part, valid_part = make_validation_split(train)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

train_part: (10272, 215)
valid_part: (3422, 215)


## Модель

Сравним несколько классических подходов на той же временной валидации:

- логистическая регрессия;
- случайный лес;
- градиентный бустинг;
- HistGradientBoostingClassifier;
- SVM;
- MLP.

Это позволит наглядно увидеть, какой алгоритм лучше справляется с задачей ранжирования и почему именно он выбран для финального решения.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import numpy as np

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_columns),
        ("cat", categorical_preprocessor, categorical_columns),
    ],
    remainder="drop",
)
# Список моделей для сравнения.

def daily_ap(y_true, y_score, dates):
    dates = pd.Series(dates)
    unique_days = sorted(dates.unique())
    aps = []
    for day in unique_days:
        mask = dates == day
        y = y_true[mask]
        s = y_score[mask]
        if len(y) == 0 or y.sum() == 0 or y.sum() == len(y):
            continue
        order = np.argsort(-s)
        y_ord = y.iloc[order]
        n_pos = int(y_ord.sum())
        if n_pos == 0:
            continue
        hits = 0
        precision_sum = 0.0
        for k, val in enumerate(y_ord, start=1):
            hits += int(val)
            precision_sum += hits / k
        aps.append(precision_sum / n_pos)
    return float(np.mean(aps)) if aps else float("nan")


models = {
    "logistic regression": LogisticRegression(max_iter=5000, class_weight="balanced"),
    "random forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "hist gradient boosting": HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_depth=6,
        max_iter=200,
        random_state=RANDOM_STATE,
    ),
    "xgboost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        scale_pos_weight=3.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=4,
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_lambda=3.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "svm": SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "mlp": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=RANDOM_STATE),
}

# Запускаем сравнение на той же временной валидации.
results = {}
predictions = {}
for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    pipeline.fit(train_part[feature_columns], train_part[TARGET])
    valid_scores = pipeline.predict_proba(valid_part[feature_columns])[:, 1]
    predictions[name] = valid_scores
    ap = average_precision_score(valid_part[TARGET], valid_scores)
    daily = daily_ap(valid_part[TARGET], valid_scores, valid_part["assignment_date"])
    results[name] = (ap, daily)
    print(f"{name}: AP={ap:.5f}, DailyAP={daily:.5f}")

ensemble_scores = 0.5 * predictions["xgboost"] + 0.5 * predictions["lightgbm"]
ensemble_ap = average_precision_score(valid_part[TARGET], ensemble_scores)
ensemble_daily = daily_ap(valid_part[TARGET], ensemble_scores, valid_part["assignment_date"])
results["xgboost+lightgbm"] = (ensemble_ap, ensemble_daily)
print(f"xgboost+lightgbm: AP={ensemble_ap:.5f}, DailyAP={ensemble_daily:.5f}")

best_name = max(results, key=lambda k: results[k][1])
print("best_by_daily_ap:", best_name)
print("best_metrics:", results[best_name])


logistic regression: AP=0.61985, DailyAP=2.03200
random forest: AP=0.58310, DailyAP=1.95199
gradient boosting: AP=0.64379, DailyAP=2.05868
hist gradient boosting: AP=0.66228, DailyAP=2.08416
xgboost: AP=0.66972, DailyAP=2.10032
[LightGBM] [Info] Number of positive: 2138, number of negative: 8134
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008530 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10006
[LightGBM] [Info] Number of data points in the train set: 10272, number of used features: 227
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.208139 -> initscore=-1.336182
[LightGBM] [Info] Start training from score -1.336182
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

c:\Users\sergei\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


svm: AP=0.58600, DailyAP=1.97365
mlp: AP=0.50126, DailyAP=1.79071
xgboost+lightgbm: AP=0.67937, DailyAP=2.11439
best_by_daily_ap: xgboost+lightgbm
best_metrics: (0.6793695680520759, 2.1143927849720394)


In [7]:
# Обучаемся на ранней части train и проверяем качество на поздней части train.
if "assignment_date" in train.columns:
    dates = pd.to_datetime(train["assignment_date"]).dt.date
    ordered_dates = sorted(dates.unique())
    cutoff = ordered_dates[int(len(ordered_dates) * 0.8)]

    train_part = train[dates < cutoff]
    valid_part = train[dates >= cutoff]
else:
    train_part, valid_part = train_test_split(
        train,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=train[TARGET],
    )

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

# Используем смесь XGBoost и LightGBM, чтобы получить более устойчивое ранжирование.
model_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.8,
            scale_pos_weight=3.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=4,
        )),
    ]
)

model_lgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.8,
            min_child_samples=20,
            reg_lambda=3.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]
)

model_xgb.fit(train_part[feature_columns], train_part[TARGET])
model_lgb.fit(train_part[feature_columns], train_part[TARGET])

xgb_scores = model_xgb.predict_proba(valid_part[feature_columns])[:, 1]
lgb_scores = model_lgb.predict_proba(valid_part[feature_columns])[:, 1]
valid_scores = 0.5 * xgb_scores + 0.5 * lgb_scores
valid_ap = average_precision_score(valid_part[TARGET], valid_scores)

print(f"Validation Average Precision: {valid_ap:.5f}")

train_part: (10272, 215)
valid_part: (3422, 215)
[LightGBM] [Info] Number of positive: 2138, number of negative: 8134
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006851 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10006
[LightGBM] [Info] Number of data points in the train set: 10272, number of used features: 227
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.208139 -> initscore=-1.336182
[LightGBM] [Info] Start training from score -1.336182
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

c:\Users\sergei\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Submission

Обучаем модель на всем train и строим score для test.  
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

In [8]:
# Финально обучаем обе модели на всей обучающей выборке и усредняем предсказания.
model_xgb.fit(train[feature_columns], train[TARGET])
model_lgb.fit(train[feature_columns], train[TARGET])

test_scores_xgb = model_xgb.predict_proba(test[feature_columns])[:, 1]
test_scores_lgb = model_lgb.predict_proba(test[feature_columns])[:, 1]
test_scores = 0.5 * test_scores_xgb + 0.5 * test_scores_lgb

submission = pd.DataFrame(
    {
        "lead_id": test["lead_id"].astype(str),
        "score": test_scores,
    }
)

submission.head()
submission.to_csv(ROOT / "submission.csv", index=False)

[LightGBM] [Info] Number of positive: 2841, number of negative: 10853
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008939 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10088
[LightGBM] [Info] Number of data points in the train set: 13694, number of used features: 227
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.207463 -> initscore=-1.340285
[LightGBM] [Info] Start training from score -1.340285
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

c:\Users\sergei\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")

submission.csv is ready
